<a href="https://colab.research.google.com/github/minmings111/practice-numpy/blob/main/sliding_window.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request

import torch
from torch.utils.data import Dataset, DataLoader

import re
from importlib.metadata import version
import tiktoken

In [2]:
url = ("http://raw.githubusercontent.com/rickiepark/"
        "llm-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

print(f"total word count : {len(raw_text)}")
print(raw_text[:99])

total word count : 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [3]:
# 입력된 기호를 기준으로 문자열 자르기
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text) # 문장 부호 + 그 뒤 공백 1칸을 한 세트로 묶어서 자르는 기준(구분자)으로 삼겠다
preprocessed = [item.strip() for item in preprocessed if item.strip()] # 공백 제거
print(len(preprocessed))

4690


In [4]:
all_words = sorted(set(preprocessed)) # 중복 제거 후, 정렬
vocab_size = len(all_words)
print(vocab_size)

1130


In [5]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
  print(item)

  if i >= 50:
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [6]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab # 다른 메서드가 참조할 수 있도록 어휘사전을 클래스의 속성으로 저장
        self.int_to_str = {i:s for s,i in vocab.items()} # 토큰 id를 원본 텍스트 토큰으로 매핑하는 역어휘사전 생성

    def encode(self, text): # 입력 텍스트를 처리해 토큰id로 변환
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids): # 토큰 아이디를 텍스트로 되돌리기
        text = " ".join([self.int_to_str[i] for i in ids])

        # 지정된 구두점 문자 앞의 공백 삭제
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [7]:
tokenizer = SimpleTokenizerV1(vocab)
text = """ "It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride. """

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [8]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


In [9]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [10]:
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i:s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [
        item.strip() for item in preprocessed if item.strip()
    ]
    preprocessed = [item if item in self.str_to_int # 모르는 단어는 unk 토큰으로 변경
                    else "<|unk|>" for item in preprocessed]

    ids = [self.str_to_int[s] for s in preprocessed]
    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # 지정된 구두점 문자 앞의 공백 삭제
    return text

In [11]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace"
text = "<|endoftext|> ".join((text1, text2))

print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>


In [12]:
#/////////////////

In [13]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace"
    " of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
print(tokenizer.decode(integers))

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 262, 20562, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace of someunknownPlace.


In [14]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [15]:
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"{x}")
print(f"     {y}")

[290, 4920, 2241, 287]
     [4920, 2241, 287, 257]


In [16]:
class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt) # 전체 텍스트 토큰화

    # 슬라이딩 윈도우를 사용해 max_length의 중첩된 시퀀스로 나눔
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunck = token_ids[i : i+max_length]
      target_chunck = token_ids[i+1 : i+max_length+1]

      self.input_ids.append(torch.tensor(input_chunck))
      self.target_ids.append(torch.tensor(target_chunck))

  # 데이터셋에 있는 전체 행 수 반환
  def __len__(self):
    return len(self.input_ids)

  # 데이터셋에서 하나의 행 반환
  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [17]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):

  tokenizer = tiktoken.get_encoding("gpt2") # tokenizer 초기화

  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride) # dataset 생성

  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                          # drop_last를 true로 설정하면, 배치 사이즈보다 작을 경우
                          # 훈련 손실이 갑자기 높아지는 것을 피하기 위해 마지막 배치를 삭제함
                          drop_last= drop_last,
                          num_workers=num_workers) # 전처리에 사용할 cpu 프로세서 수

  return dataloader

In [20]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

# dataloader를 반복자로 변환 후, 다음 원소 추출
data_iter = iter(dataloader)

first_batch = next(data_iter)
print(first_batch)

second_batch = next(data_iter)
print(second_batch)

# print(data_iter)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [26]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

In [32]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print(embedding_layer)
print(embedding_layer.weight)
print(embedding_layer((torch.tensor([3]))))
print(embedding_layer(input_ids))

Embedding(6, 3)
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [35]:
# 임베딩

vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [39]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("token id \n", inputs)

print("input_size \n", inputs.shape)

token id 
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
input_size 
 torch.Size([8, 4])


In [40]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [41]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))

print(pos_embeddings.shape)

torch.Size([4, 256])


In [43]:
input_embeddings = token_embeddings + pos_embeddings

print(input_embeddings.shape)

torch.Size([8, 4, 256])
